In [ ]:
import glob
import os
import re

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC

META_COLS = ['sample', 'binary', 'disease']
STATS = ['pfe', 'lwps', 'ifs', 'fdi', 'ocf', 'cnn', 'mlp', 'vae']
MAX_SAMPLE_LOSS = 0.2


font_path = './AUPassata_Rg.ttf'
try:
    fm.fontManager.addfont(font_path)
except FileNotFoundError:
    print(f'Font not found at {font_path}, falling back to default font.')


def init_plotting():
    plt.rcParams['figure.figsize'] = (8, 3)
    plt.rcParams['font.size'] = 10
    plt.rcParams['font.family'] = 'AU Passata'
    plt.rcParams['axes.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['axes.titlesize'] = 1.5 * plt.rcParams['font.size']
    plt.rcParams['legend.fontsize'] = plt.rcParams['font.size']
    plt.rcParams['xtick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['ytick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['savefig.dpi'] = 200
    plt.rcParams['xtick.major.size'] = 3
    plt.rcParams['xtick.major.width'] = 1
    plt.rcParams['ytick.major.size'] = 3
    plt.rcParams['ytick.major.width'] = 1
    plt.rcParams['legend.frameon'] = False
    plt.rcParams['axes.linewidth'] = 1


init_plotting()


with open('../confs/thesis.yaml') as f:
    config = yaml.safe_load(f)

FINAL_MATRICES_DIR = config['data']['final_matrices_dir']
# FINAL_MATRICES_DIR = 'previous_data/final_matrices/'
ACCESSIBILITY_DIR = config['data']['accessibility_scores_dir']
# ACCESSIBILITY_DIR = 'previous_data/accessibility_scores/'
MIN_COV_FILE = f'../{config["data"]["training_min_coverage_file"]}'
# MIN_COV_FILE = '../previous_data/training_frags/min_coverage.txt'

with open(MIN_COV_FILE) as f:
    COVERAGE_THRESHOLD = int(f.read().strip())


def load_score_dfs(stats, final_matrices_dir):
    score_dfs = {}
    for stat in stats:
        path = os.path.join(f'../{final_matrices_dir}', f'feature_matrix_{stat}.parquet')
        if os.path.exists(path):
            score_dfs[stat] = pd.read_parquet(path)
            print(f'{stat}: {score_dfs[stat].shape}')
        else:
            print(f'{stat}: not found')
    return score_dfs


def load_coverage_df(accessibility_dir):
    cov_files = sorted(glob.glob(os.path.join(f'../{accessibility_dir}', '*.cov.txt')))

    records = []
    for path in cov_files:
        basename = os.path.basename(path)
        match = re.match(r'(.+?)__(.+?)\.cov\.txt$', basename)
        if not match:
            continue

        sample, dhs = match.group(1), match.group(2)
        with open(path) as f:
            cov = int(f.read().strip())

        records.append({'sample': sample, 'dhs': dhs, 'coverage': cov})

    cov_df = pd.DataFrame(records).pivot(index='sample', columns='dhs', values='coverage')
    cov_df.columns = pd.MultiIndex.from_product([['coverage'], cov_df.columns])
    return cov_df


def attach_coverage(score_dfs, cov_df):
    for stat, df in score_dfs.items():
        samples = df.index.get_level_values('sample')
        cov_aligned = cov_df.loc[samples]
        cov_aligned.index = df.index
        score_dfs[stat] = pd.concat([df, cov_aligned], axis=1)
    return score_dfs


def _filter_to_samples(score_dfs, valid_samples):
    for stat, df in score_dfs.items():
        mask = df.index.get_level_values('sample').isin(valid_samples)
        score_dfs[stat] = df.loc[mask]


def qc_filter(score_dfs):
    sample_sets = []
    for _, df in score_dfs.items():
        sample_sets.append(set(df.index.get_level_values('sample')))

    common_samples = set.intersection(*sample_sets)
    _filter_to_samples(score_dfs, common_samples)
    print(f'initial samples: {len(common_samples)}')

    common_keep_dhs = None
    for _, df in score_dfs.items():
        coverages = df.xs('coverage', axis=1, level=0)
        n_samples = coverages.shape[0]
        missing_fraction = (coverages < COVERAGE_THRESHOLD).sum(axis=0) / n_samples
        keep_dhs = missing_fraction <= MAX_SAMPLE_LOSS

        if common_keep_dhs is None:
            common_keep_dhs = keep_dhs
        else:
            common_keep_dhs &= keep_dhs

    dropped = (~common_keep_dhs).sum()
    print(f'dropped {dropped} DHS columns globally')

    kept_cols = common_keep_dhs[common_keep_dhs].index
    for stat, df in score_dfs.items():
        score_dfs[stat] = df[df.columns[df.columns.get_level_values(1).isin(kept_cols)]]

    valid_samples = None
    for _, df in score_dfs.items():
        coverage = df.xs('coverage', axis=1, level=0)
        good = (coverage >= COVERAGE_THRESHOLD).all(axis=1)
        samples = set(df.index.get_level_values('sample')[good])

        if valid_samples is None:
            valid_samples = samples
        else:
            valid_samples &= samples

    print(f'samples after coverage filter: {len(valid_samples)}')
    _filter_to_samples(score_dfs, valid_samples)

    valid_samples = None
    for _, df in score_dfs.items():
        features = df.xs('pc1', axis=1, level=0)
        good = ~features.isna().any(axis=1)
        samples = set(df.index.get_level_values('sample')[good])

        if valid_samples is None:
            valid_samples = samples
        else:
            valid_samples &= samples

    print(f'samples after missing-score filter: {len(valid_samples)}')
    _filter_to_samples(score_dfs, valid_samples)
    return score_dfs


def extract_feature_df(df, feature='pc1'):
    meta = df.index.to_frame()[META_COLS].reset_index(drop=True)
    features = df.xs(feature, axis=1, level=0)
    out = pd.concat([meta, features.reset_index(drop=True)], axis=1)
    return out.dropna(axis=1, how='any')


def merge_stats(dfs, feature='pc1'):
    df_all = None
    for stat, df in dfs.items():
        extracted = extract_feature_df(df, feature)
        feature_cols = [c for c in extracted.columns if c not in META_COLS]
        renamed = extracted.rename(columns={c: f'{c}_{stat}' for c in feature_cols})

        if df_all is None:
            df_all = renamed
        else:
            df_all = pd.merge(df_all, renamed, on=META_COLS, how='inner')
    return df_all


def evaluate_stat_joint(stat, dfs, feature='pc1', cv_splits=10):
    if stat == 'all':
        df = merge_stats(dfs, feature)
        stat = '+'.join(dfs.keys())
    else:
        df = extract_feature_df(dfs[stat], feature)

    labels = df['binary']
    y = LabelEncoder().fit_transform(labels)

    feature_cols = [c for c in df.columns if c not in META_COLS]
    X = df[feature_cols].to_numpy()

    clf = SVC(probability=True, random_state=42)
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    scores = cross_val_score(
        clf,
        X,
        y,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1,
    )

    y_pred_proba = cross_val_predict(
        clf,
        X,
        y,
        cv=cv,
        method='predict_proba',
        n_jobs=-1,
    )[:, 1]

    auc_proba = roc_auc_score(y, y_pred_proba)

    return {
        'stat': stat,
        'n_features': X.shape[1],
        'auc_mean': float(np.mean(scores)),
        'auc_std': float(np.std(scores)),
        'auc_proba_overall': auc_proba,
        'y_true': y,
        'y_pred_proba': y_pred_proba,
    }


def visualize_coverage(scores, stat='vae'):
    cov = scores[stat]['coverage'].reset_index()
    cov_long = cov.melt(id_vars=META_COLS, var_name='DHS', value_name='coverage')

    groups = [g['coverage'].values for _, g in cov_long.groupby('DHS')]
    labels = cov_long['DHS'].unique()

    plt.figure(figsize=(12, 6))
    plt.boxplot(groups, showmeans=True)
    plt.xticks(range(1, len(labels) + 1), labels, rotation=45, ha='right')
    plt.ylabel('Coverage')
    plt.title(f'Coverage distribution per DHS ({stat})')
    plt.tight_layout()
    plt.show()


score_dfs = load_score_dfs(STATS, FINAL_MATRICES_DIR)
cov_df = load_coverage_df(ACCESSIBILITY_DIR)
score_dfs = attach_coverage(score_dfs, cov_df)

In [ ]:
# coverage before filtering
visualize_coverage(score_dfs)

In [ ]:
score_dfs = qc_filter(score_dfs)

for stat, df in score_dfs.items():
    features = df.xs('pc1', axis=1, level=0)
    print(stat, features.shape)

results = []
dfs = {k: v for k, v in score_dfs.items() if k in STATS}

for stat in dfs:
    result = evaluate_stat_joint(stat, dfs, feature='pc1')
    if result is not None:
        results.append(result)

# optional combined model
# if dfs:
#     results.append(evaluate_stat_joint('all', dfs, feature='pc1'))

In [ ]:
# coverage after filtering
visualize_coverage(score_dfs)

In [ ]:
if results:
    stats_order = [r['stat'] for r in results]
    auc_proba_overall_values = [r['auc_proba_overall'] for r in results]

    plt.figure(figsize=(8, 6))
    bars = plt.bar(stats_order, auc_proba_overall_values, capsize=5)
    plt.ylabel('Overall ROC AUC proba (all DHS sites)')
    plt.xlabel('Test statistic')

    for bar, val in zip(bars, auc_proba_overall_values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f'{val:.2f}',
            ha='center',
            va='bottom',
            fontsize=9,
        )

    plt.title('Binary classification using joint DHS PC1 features + SVM')
    plt.tight_layout()
    out_path_classification = os.path.join(
        FINAL_MATRICES_DIR,
        'classification_pc1_auc_by_stat_overall_auc_proba.png',
    )
    # plt.savefig(out_path_classification, dpi=200)
    plt.close()
    plt.show()

    plt.figure(figsize=(8, 6))
    for r in results:
        fpr, tpr, _ = roc_curve(r['y_true'], r['y_pred_proba'])
        plt.plot(fpr, tpr, label=f'{r["stat"]} (AUC={r["auc_proba_overall"]:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC curves for all statistics (cross-validated predictions) Coverage')
    plt.legend()
    plt.tight_layout()
    out_path_classification = os.path.join(FINAL_MATRICES_DIR, 'roc_curves_pc1_all_stats.png')
    plt.show()
    # plt.savefig(out_path_classification, dpi=200)
    plt.close()